# Can we predict `passed_quiz`?
Follows `checklist.md`. Key facts about the data (from `columns.csv` and inspection):
- 12,453 session rows, 1,800 students (1-17 rows each) -> **whole students** stay together in every split.
- There is no session date/time column (`enrol_year` is constant per student), so a "later rows" split is not possible; grouped splits are the right structure.
- `student_key` is an identifier: dropped. `enrol_year` and `school_code` are constant per student.
- Assumption: prediction is made when the closing quiz starts, so `session_minutes` and `hints_used` (recorded during the session) are available. If predictions are needed before the session, those two must be removed.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
df=pd.read_csv('data.csv')
print(df.shape, df.passed_quiz.mean())

## 1. Hold out a test set (whole students) before any modelling

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=0)
tr,te=next(gss.split(df,df.passed_quiz,df.student_key))
train,test=df.iloc[tr].reset_index(drop=True),df.iloc[te].reset_index(drop=True)
assert not set(train.student_key)&set(test.student_key)
print(len(train),len(test),train.passed_quiz.mean(),test.passed_quiz.mean())
# The test set is not touched again until the final step.

## 2. Candidate models (named before any scoring)
Features exclude `student_key`.
1. **baseline**: predicts training prevalence (constant).
2. **logit_prior**: logistic regression on `prior_score` only.
3. **logit_session**: logistic regression on `prior_score, session_minutes, hints_used`.
4. **logit_all**: as 3 plus `enrol_year` and one-hot `school_code`.
5. **hgb_session**: gradient boosting on `prior_score, session_minutes, hints_used`.
6. **hgb_all**: gradient boosting on those plus `enrol_year, school_code`.

Selection metric: mean ROC AUC over grouped CV folds in the training rows. Decision threshold: maximises mean CV F1 on out-of-fold predictions. Both rules are fixed here, before scoring.

In [ ]:
S=['prior_score','session_minutes','hints_used']
def logit(cols,cat=()):
    pre=ColumnTransformer([('n',StandardScaler(),[c for c in cols if c not in cat])]+
        ([('c',OneHotEncoder(handle_unknown='ignore'),list(cat))] if cat else []))
    return cols,make_pipeline(pre,LogisticRegression(max_iter=1000))
def hgb(cols): return cols,HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)
cands={
 'baseline':(['prior_score'],DummyClassifier(strategy='prior')),
 'logit_prior':logit(['prior_score']),
 'logit_session':logit(S),
 'logit_all':logit(S+['enrol_year','school_code'],cat=['school_code']),
 'hgb_session':hgb(S),
 'hgb_all':hgb(S+['enrol_year','school_code']),
}

## 3. Grouped cross-validation on training rows only
Scaler/encoder live inside the pipelines, so preprocessing is fit within each fold.

In [ ]:
ytr=train.passed_quiz.values; gtr=train.student_key.values
folds=list(GroupKFold(n_splits=5).split(train,ytr,gtr))
oof={}; rows=[]
for name,(cols,m) in cands.items():
    p=np.zeros(len(train)); aucs=[]
    for a,b in folds:
        mm=clone(m).fit(train.iloc[a][cols],ytr[a])
        p[b]=mm.predict_proba(train.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(ytr[b],p[b]))
    oof[name]=p; rows.append((name,np.mean(aucs),np.std(aucs)))
cv=pd.DataFrame(rows,columns=['model','cv_auc','sd']).sort_values('cv_auc',ascending=False)
print(cv)

## 4. Choose model and threshold on CV only

In [ ]:
best=cv.iloc[0].model
ths=np.linspace(0.05,0.95,91)
f1s=[np.mean([f1_score(ytr[b],oof[best][b]>t) for a,b in folds]) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'threshold:',thr,'CV F1:',max(f1s))

**Decision (recorded before scoring the test set):** the model and threshold printed above are final. The model is refit on all training rows and scored once on the test set.

## 5. Final: refit on all training rows, score test once

In [ ]:
cols,m=cands[best]
final=clone(m).fit(train[cols],ytr)
pt=final.predict_proba(test[cols])[:,1]; yt=test.passed_quiz.values
auc=roc_auc_score(yt,pt)
rng=np.random.default_rng(0); ids=test.student_key.values; u=np.unique(ids)
idx={s:np.where(ids==s)[0] for s in u}; bs=[]
for _ in range(500):
    ii=np.concatenate([idx[s] for s in rng.choice(u,len(u))])
    if len(np.unique(yt[ii]))>1: bs.append(roc_auc_score(yt[ii],pt[ii]))
print(f'TEST AUC {auc:.3f}  95% student-bootstrap CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f}')
print('TEST accuracy',accuracy_score(yt,pt>thr),'F1',f1_score(yt,pt>thr),'| majority-class accuracy',max(yt.mean(),1-yt.mean()))

## Conclusion
Compare the headline test AUC (with CI) to the baseline's 0.5. Caveats: no timestamp, so generalisation to later sessions cannot be checked; `session_minutes`/`hints_used` assume prediction at quiz start.